In [2]:
!pip install transformers torch


In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [4]:
# Load DialoGPT model - specifically designed for conversations
model_name = "microsoft/DialoGPT-medium"

# Load tokenizer - converts text to tokens that model understands
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load the pre-trained model
print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(model_name)

print(f"\nModel '{model_name}' loaded successfully!")
print(f"Model parameters: {model.num_parameters():,}")

Loading tokenizer...


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Loading model...


pytorch_model.bin:   0%|          | 0.00/863M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]


Model 'microsoft/DialoGPT-medium' loaded successfully!
Model parameters: 354,823,168


In [5]:
sample_text = "Hello, how are you?"

# Encode text to token IDs
encoded = tokenizer.encode(sample_text, return_tensors="pt")
print(f"Original text: {sample_text}")
print(f"Token IDs: {encoded}")
print(f"Individual tokens: {tokenizer.tokenize(sample_text)}")

# Decode back to text
decoded = tokenizer.decode(encoded[0], skip_special_tokens=True)
print(f"Decoded text: {decoded}")

Original text: Hello, how are you?
Token IDs: tensor([[15496,    11,   703,   389,   345,    30]])
Individual tokens: ['Hello', ',', 'Ġhow', 'Ġare', 'Ġyou', '?']
Decoded text: Hello, how are you?


In [6]:
def generate_response(user_input, chat_history_ids=None, max_length=1000):

    new_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,  # Add EOS token
        return_tensors="pt"                 # Return PyTorch tensor
    )
    
    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        bot_input_ids = new_input_ids
    
    chat_history_ids = model.generate(
        bot_input_ids,
        max_length=max_length,
        pad_token_id=tokenizer.eos_token_id,
        no_repeat_ngram_size=3,       # Avoid repeating 3-word phrases
        do_sample=True,                # Enable sampling for diverse responses
        top_k=50,                      # Consider top 50 tokens
        top_p=0.95,                    # Nucleus sampling threshold
        temperature=0.75               # Control randomness (lower = focused)
    )
    
    response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )
    
    return response, chat_history_ids

test_response, test_history = generate_response("Hello, how are you?")
print(f"User: Hello, how are you?")
print(f"Bot: {test_response}")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


model.safetensors:   0%|          | 0.00/863M [00:00<?, ?B/s]

User: Hello, how are you?
Bot: I'm good thanks


In [7]:
test_inputs = [
    "What is Artificial Intelligence?",
    "Tell me a joke",
    "What is Python programming?",
    "Who are you?"
]

print("=" * 60)
print("TESTING INDIVIDUAL RESPONSES")
print("=" * 60)

for user_input in test_inputs:
    response, _ = generate_response(user_input)
    print(f"\nUser: {user_input}")
    print(f"Bot:  {response}")
    print("-" * 40)
    

TESTING INDIVIDUAL RESPONSES

User: What is Artificial Intelligence?
Bot:  Baby don't hurt me
----------------------------------------

User: Tell me a joke
Bot:  What does the pope have to do with this?
----------------------------------------

User: What is Python programming?
Bot:  I don't know
----------------------------------------

User: Who are you?
Bot:  I'm me, and you're me!
----------------------------------------


In [8]:
def chatbot():
    chat_history_ids = None
    step_count = 0  # Track conversation turns
    print("=" * 60)
    print("       AI CHATBOT - Powered by DialoGPT")
    print("=" * 60)
    print("Chatbot: Hello! I am your AI assistant.")
    print("         How can I help you today?")
    print("         (Type 'exit' or 'quit' to end)")
    print("=" * 60)
    
    while True:
        user_input = input("\nYou: ").strip()
        
        if not user_input:
            print("Chatbot: Please type something. I'm here to help!")
            continue
        
        if user_input.lower() in ['exit', 'quit', 'bye', 'goodbye']:
            print("\nChatbot: Goodbye! It was nice talking to you.")
            print("         Have a great day! 👋")
            print("=" * 60)
            break
        
        try:
            response, chat_history_ids = generate_response(
                user_input, 
                chat_history_ids
            )
            step_count += 1
            
            if step_count > 5:
                chat_history_ids = None
                step_count = 0
            
            print(f"Chatbot: {response}")
            
        except Exception as e:
            print(f"Chatbot: I'm sorry, I encountered an error. Let me reset.")
            print(f"         Error: {str(e)}")
            chat_history_ids = None
            step_count = 0

chatbot()

       AI CHATBOT - Powered by DialoGPT
Chatbot: Hello! I am your AI assistant.
         How can I help you today?
         (Type 'exit' or 'quit' to end)



You:  exit



Chatbot: Goodbye! It was nice talking to you.
         Have a great day! 👋


In [9]:
from datetime import datetime

def enhanced_chatbot():
    
    chat_history_ids = None
    step_count = 0
    conversation_log = []  # Store conversation for display
    
    print("\n" + "=" * 60)
    print("    🤖 ENHANCED AI CHATBOT - DialoGPT Medium")
    print("=" * 60)
    print("Chatbot: Hello! I am your AI assistant powered by")
    print("         Microsoft DialoGPT.")
    print("         How can I help you today?")
    print("\n📌 Commands:")
    print("   • Type 'exit' or 'quit' to end conversation")
    print("   • Type 'history' to view conversation log")
    print("   • Type 'reset' to clear conversation context")
    print("=" * 60)
    
    while True:
        user_input = input("\n👤 You: ").strip()
        
        if not user_input:
            print("🤖 Chatbot: I didn't catch that. Could you please type something?")
            continue
        
        if user_input.lower() in ['exit', 'quit', 'bye', 'goodbye']:
            print("\n🤖 Chatbot: Goodbye! Thanks for chatting with me! 👋")
            
            print(f"\n📊 Conversation Summary:")
            print(f"   Total exchanges: {len(conversation_log)}")
            print("=" * 60)
            break
        
        if user_input.lower() == 'history':
            print("\n📜 Conversation History:")
            print("-" * 40)
            if conversation_log:
                for entry in conversation_log:
                    print(f"   👤 You: {entry['user']}")
                    print(f"   🤖 Bot: {entry['bot']}")
                    print(f"   ⏰ Time: {entry['time']}")
                    print("-" * 40)
            else:
                print("   No conversation history yet.")
            continue
        
        if user_input.lower() == 'reset':
            chat_history_ids = None
            step_count = 0
            print("🤖 Chatbot: Conversation context has been reset!")
            continue
        
        try:
            response, chat_history_ids = generate_response(
                user_input, 
                chat_history_ids
            )
            step_count += 1
            
            conversation_log.append({
                'user': user_input,
                'bot': response,
                'time': datetime.now().strftime("%H:%M:%S")
            })
            
            if step_count > 5:
                chat_history_ids = None
                step_count = 0
            
            print(f"🤖 Chatbot: {response}")
            
        except Exception as e:
            print(f"🤖 Chatbot: Sorry, let me try again. (Resetting context)")
            chat_history_ids = None
            step_count = 0

enhanced_chatbot()


    🤖 ENHANCED AI CHATBOT - DialoGPT Medium
Chatbot: Hello! I am your AI assistant powered by
         Microsoft DialoGPT.
         How can I help you today?

📌 Commands:
   • Type 'exit' or 'quit' to end conversation
   • Type 'history' to view conversation log
   • Type 'reset' to clear conversation context



👤 You:  exit



🤖 Chatbot: Goodbye! Thanks for chatting with me! 👋

📊 Conversation Summary:
   Total exchanges: 0


In [10]:
print("=" * 60)
print("UNDERSTANDING GENERATION PARAMETERS")
print("=" * 60)

test_input = "What is the meaning of life?"

print("\n1️⃣ LOW TEMPERATURE (0.3) - More focused/deterministic:")
input_ids = tokenizer.encode(test_input + tokenizer.eos_token, return_tensors="pt")
output = model.generate(
    input_ids,
    max_length=100,
    pad_token_id=tokenizer.eos_token_id,
    do_sample=True,
    temperature=0.3,
    top_k=50
)
response = tokenizer.decode(output[:, input_ids.shape[-1]:][0], skip_special_tokens=True)
print(f"   User: {test_input}")
print(f"   Bot:  {response}")

# Parameter set 2: Creative (High temperature)
print("\n2️⃣ HIGH TEMPERATURE (1.0) - More creative/random:")
output = model.generate(
    input_ids,
    max_length=100,
    pad_token_id=tokenizer.eos_token_id,
    do_sample=True,
    temperature=1.0,
    top_k=50
)
response = tokenizer.decode(output[:, input_ids.shape[-1]:][0], skip_special_tokens=True)
print(f"   User: {test_input}")
print(f"   Bot:  {response}")

print("\n3️⃣ BEAM SEARCH (num_beams=5) - More coherent:")
output = model.generate(
    input_ids,
    max_length=100,
    pad_token_id=tokenizer.eos_token_id,
    num_beams=5,
    no_repeat_ngram_size=3
)
response = tokenizer.decode(output[:, input_ids.shape[-1]:][0], skip_special_tokens=True)
print(f"   User: {test_input}")
print(f"   Bot:  {response}")

print("\n" + "=" * 60)

UNDERSTANDING GENERATION PARAMETERS

1️⃣ LOW TEMPERATURE (0.3) - More focused/deterministic:
   User: What is the meaning of life?
   Bot:  What is love?

2️⃣ HIGH TEMPERATURE (1.0) - More creative/random:
   User: What is the meaning of life?
   Bot:  Happiness is the only fulfillment

3️⃣ BEAM SEARCH (num_beams=5) - More coherent:
   User: What is the meaning of life?
   Bot:  What is love?



In [11]:
print("=" * 60)
print("MODEL ARCHITECTURE SUMMARY")
print("=" * 60)

print(f"\n📌 Model Name: {model_name}")
print(f"📌 Model Type: {model.config.model_type}")
print(f"📌 Number of Parameters: {model.num_parameters():,}")
print(f"📌 Number of Layers: {model.config.n_layer}")
print(f"📌 Hidden Size: {model.config.n_embd}")
print(f"📌 Number of Attention Heads: {model.config.n_head}")
print(f"📌 Vocabulary Size: {model.config.vocab_size}")
print(f"📌 Max Position Embeddings: {model.config.n_positions}")

print("\n" + "=" * 60)
print("PIPELINE FLOW")
print("=" * 60)

MODEL ARCHITECTURE SUMMARY

📌 Model Name: microsoft/DialoGPT-medium
📌 Model Type: gpt2
📌 Number of Parameters: 354,823,168
📌 Number of Layers: 24
📌 Hidden Size: 1024
📌 Number of Attention Heads: 16
📌 Vocabulary Size: 50257
📌 Max Position Embeddings: 1024

PIPELINE FLOW
